In [ ]:
# OutfitMatch Stylist - GRPO RL (verifiable rewards) on Kaggle T4 (16GB)
# Logs ONLINE to W&B. API keys embedded per project policy (.env.local).
import os, sys, subprocess, time, traceback

os.environ['KAGGLE_API_KEY'] = 'KGAT_ed6e5c83d4481d9ebeb9520b1726f14d'
os.environ['WANDB_API_KEY'] = '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'
os.environ['WANDB_MODE'] = 'online'
os.environ['WANDB_PROJECT'] = 'outfitmatch-stylist'
os.environ['WANDB_ENTITY'] = 'vominhnhatquang-fpt-university'
os.environ['WANDB_SILENT'] = 'true'
os.environ['HF_HUB_CACHE'] = '/kaggle/working/hf_cache'
os.environ['HF_HOME'] = '/kaggle/working/hf_home'
os.environ['BITSANDBYTES_NOWELCOME'] = '1'
os.environ['PYTHONPATH'] = '/kaggle/working/fashion_match/src' + os.pathsep + os.environ.get('PYTHONPATH', '')
print('secrets + env configured (WANDB_MODE=%s)' % os.environ['WANDB_MODE'])


In [ ]:
# 1) Dep install. Pin EXACTLY the local working stack:
#    transformers==5.8.1 (contains Qwen3-VL rope_deltas GRPO fix, PR #44873),
#    trl==0.14.0 (GRPOTrainer contract), compatible peft/accelerate/huggingface-hub.
#    force-reinstall --no-deps keeps Kaggle's torch.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
                       'trl==0.14.0', 'transformers==5.8.1', 'accelerate==1.13.0',
                       'bitsandbytes', 'peft==0.19.1', 'datasets', 'wandb', 'huggingface-hub==1.15.0'])
# trl 0.14 imports `from vllm import LLM, SamplingParams` at top-level behind a broken
# is_vllm_available() tuple. Stub it so the import succeeds; GRPO uses use_vllm=False.
import os
os.makedirs('/kaggle/working/vllm_stub/vllm', exist_ok=True)
with open('/kaggle/working/vllm_stub/vllm/__init__.py', 'w') as f:
    f.write('class LLM:\n')
    f.write('    def __init__(self, *a, **k):\n')
    f.write('        raise RuntimeError("vllm disabled; GRPO uses HF generate()")\n')
    f.write('class SamplingParams:\n')
    f.write('    def __init__(self, *a, **k):\n')
    f.write('        raise RuntimeError("vllm disabled")\n')
print('vllm stub written')
sys.path.insert(0, '/kaggle/working/vllm_stub')
print('deps installed + vllm stub added')


In [ ]:
# 2) Fetch code + IDEMPOTENT patch so the training always works (safety net).
#    The branch HEAD already carries the fixes; these patches cover any stale clone.
REPO = 'https://github.com/Lendo-Stylix/fashion_match.git'
BRANCH = 'feat/benchmark'
root = '/kaggle/working/fashion_match'
if not os.path.isdir(os.path.join(root, '.git')):
    subprocess.check_call(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, root])
script = os.path.join(root, 'scripts', 'stylist', 'train_grpo_kaggle.py')
src = open(script, encoding='utf-8').read()
anchor1 = '    parser.add_argument("--push-to-hub", default=None, help="HF repo id to push adapter")'
new_args = anchor1 + chr(10) + '    parser.add_argument("--report-to", default="none", help="W&B logger target (wandb/none).")' + chr(10) + '    parser.add_argument("--wandb-project", default="outfitmatch-stylist")' + chr(10) + '    parser.add_argument("--wandb-run-name", default=None)'
if 'report-to' not in src:
    assert anchor1 in src, 'clone missing push-to-hub anchor'
    src = src.replace(anchor1, new_args)
anchor2 = '        seed=args.seed,'
new_cfg = '        seed=args.seed,' + chr(10) + '        report_to=args.report_to,' + chr(10) + '        run_name=args.wandb_run_name,'
if 'report_to=args.report_to' not in src:
    assert anchor2 in src, 'clone missing seed anchor'
    src = src.replace(anchor2, new_cfg)
anchor3 = '        low_cpu_mem_usage=True,\n    )'
new_load = '        low_cpu_mem_usage=True,\n        attn_implementation="eager",\n    )'
if 'attn_implementation="eager"' not in src:
    assert anchor3 in src, 'clone missing model load anchor'
    src = src.replace(anchor3, new_load)
open(script, 'w', encoding='utf-8').write(src)
print('patched train script ok:', 'report-to' in src and 'report_to=args.report_to' in src and 'attn_implementation="eager"' in src)


In [ ]:
# 3) Sanity: import reward module (verifies outfitmatch + scorers load) and build dataset.
import sys
sys.path.insert(0, '/kaggle/working/fashion_match/src')
sys.path.insert(0, '/kaggle/working/fashion_match/scripts/stylist')
from grpo_rewards import FASHION_REWARD_FUNCS, build_fashion_prompt_pool
pool = build_fashion_prompt_pool()
print('reward funcs:', [f.__name__ for f in FASHION_REWARD_FUNCS])
print('pool size:', len(pool), 'types:', sorted({r['item_type'] for r in pool}))


In [ ]:
# 4) GRPO RL run. Qwen3-VL 8B bnb-4bit barely fits a single T4 16GB, so use a SMART conservative
#    config: batch=1, G=2 (only 2 seqs/step), completion 128 tokens. This is a real GRPO run
#    with verifiable rewards + online W&B logging (just shorter than the 500-step production run).
#    Override with env: GRPO_BATCH / GRPO_GENS / GRPO_MAX_STEPS / GRPO_LEN.
#    --report-to wandb -> trl/transformers call wandb.init() ONLINE (WANDB_API_KEY set).
import runpy
MAX_STEPS = int(os.environ.get('GRPO_MAX_STEPS', '200'))
BATCH = int(os.environ.get('GRPO_BATCH', '1'))
GENS = int(os.environ.get('GRPO_GENS', '2'))
MCLEN = int(os.environ.get('GRPO_LEN', '128'))
sys.path.insert(0, '/kaggle/working/fashion_match')
sys.path.insert(0, '/kaggle/working/fashion_match/scripts/stylist')
argv = ['train_grpo_kaggle.py',
        '--model', 'unsloth/Qwen3-VL-8B-Instruct-bnb-4bit',
        '--report-to', 'wandb',
        '--wandb-project', 'outfitmatch-stylist',
        '--wandb-run-name', 'om-grpo-t4-instruct',
        '--max-steps', str(MAX_STEPS),
        '--num-generations', str(GENS),
        '--batch-size', str(BATCH),
        '--max-completion-length', str(MCLEN),
        '--temperature', '0.7',
        '--lora-r', '32',
        '--think',
        '--output-dir', '/kaggle/working/outputs/grpo-t4-instruct']
sys.argv = argv
print('Running:', ' '.join(argv))
t0 = time.time()
try:
    runpy.run_path('/kaggle/working/fashion_match/scripts/stylist/train_grpo_kaggle.py', run_name='__main__')
except SystemExit as e:
    print('train exited with code', e.code)
    raise
print('GRPO finished in %.1f min' % ((time.time() - t0) / 60))


In [ ]:
# 5) Persist artifacts: zip the LoRA adapter so it appears in notebook output files (downloadable).
import shutil
out_root = '/kaggle/working/outputs/grpo-t4-instruct'
if os.path.isdir(out_root):
    zip_path = '/kaggle/working/grpo-t4-instruct-adapter.zip'
    shutil.make_archive(zip_path[:-4], 'zip', out_root)
    print('adapter zipped ->', zip_path, 'size MB=%.1f' % (os.path.getsize(zip_path)/1e6))
    print('adapter files:', sorted(os.listdir(out_root))[:20])
else:
    print('WARNING: output dir missing, run may not have completed')
